<a href="https://colab.research.google.com/github/jzwillucf/Hetnet_Analysis/blob/main/PubMed_Pull_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiate Neo4j

In [ ]:
pip install neo4j

In [ ]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable, AuthError
import pandas as pd

uri = "blank"
user = "neo4j"
password = "blank"

In [ ]:
try:
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        driver.verify_connectivity()
        print("Connection to Neo4j database successful!")
except Exception as e:
    print(f"Connection failed: {e}")

#Paper Pulling

In [ ]:
query = """
MATCH ()-[r]->()
WHERE r.PubMedIDs IS NOT NULL
RETURN type(r) AS RelationshipType, r.PubMedIDs AS PubMedIDs, startNode(r) AS StartNode, endNode(r) AS EndNode
LIMIT 100
"""

try:
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        with driver.session() as session:
            result = session.run(query)
            records = list(result)
            found = False
            for record in records:
                val = record['PubMedIDs']
                # If it is a list/array with more than 1 item, or a string containing multiple IDs split by commas/semicolons
                if (isinstance(val, list) and len(val) > 1) or (isinstance(val, str) and (',' in val or ';' in val)):
                    print(f"Relationship: {record['RelationshipType']}")
                    print(f"  PubMedIDs: {val}")
                    print(f"  From Node: {record['StartNode']}")
                    print(f"  To Node: {record['EndNode']}\n")
                    found = True
                    break
            if not found:
                print("No relationships found with multiple PubMedIDs within the fetched sample.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import requests
import xml.etree.ElementTree as ET

if isinstance(val, list):
    pmids = ",".join(val)
elif isinstance(val, str):
    pmids = val.replace(";", ",")
else:
    pmids = ""

if pmids:
    print(f"Fetching PubMed data for PMIDs: {pmids}\n")

    # Fetch metadata and abstract using efetch
    efetch_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id={pmids}&retmode=xml"
    try:
        response = requests.get(efetch_url)
        response.raise_for_status()
        root = ET.fromstring(response.content)

        for article in root.findall('.//PubmedArticle'):
            pmid_node = article.find('.//PMID')
            pmid = pmid_node.text if pmid_node is not None else "Unknown"

            title_node = article.find('.//ArticleTitle')
            title = title_node.text if title_node is not None else "No Title"

            abstract_texts = article.findall('.//AbstractText')
            abstract = " ".join([elem.text for elem in abstract_texts if elem.text]) if abstract_texts else "No abstract available"

            pmcid = None
            for article_id in article.findall('.//ArticleId'):
                if article_id.attrib.get('IdType') == 'pmc':
                    pmcid = article_id.text

            print(f"PMID: {pmid}")
            print(f"Title: {title}")
            print(f"Abstract: {abstract}\n")

            if pmcid:
                print(f"Found {pmcid}. Fetching full text from PubMed Central...")
                pmc_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&id={pmcid}&retmode=xml"
                pmc_response = requests.get(pmc_url)
                if pmc_response.status_code == 200:
                    try:
                        pmc_root = ET.fromstring(pmc_response.content)
                        paragraphs = pmc_root.findall('.//body//p')
                        full_text = "\n".join([p.text for p in paragraphs if p.text])
                        if full_text:
                            print(f"Full Text Snippet (first 1000 chars):\n{full_text[:1000]}...\n")
                            print(f"(Read full paper at: https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/)\n")
                        else:
                            print("Full text body not parsed correctly or empty.\n")
                    except ET.ParseError:
                        print("Error parsing PMC XML.\n")
                else:
                    print(f"Failed to fetch from PMC. Status code: {pmc_response.status_code}\n")
            else:
                print("No PMCID found. Full paper is likely not available in the open access PubMed Central repository.\n")

            print("=" * 80 + "\n")

    except Exception as e:
        print(f"Error fetching data: {e}")
else:
    print("No PMIDs found to look up.")

In [ ]:
pip install biopython

In [ ]:
from Bio import Entrez
import xml.etree.ElementTree as ET

# NCBI requires an email address to track usage and notify you if you exceed rate limits
Entrez.email = "jo417685@ucf.edu"
# API key will increaase your rate limit
Entrez.api_key = "28e0c0a8f448c4d2a64a96d984a4cb9e6b09"

def get_pubmed_metadata(pmids):
    # 1. Fetch metadata using efetch
    handle = Entrez.efetch(db="pubmed", id=",".join(pmids), retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    # Fetch citation counts using elink
    citation_counts = {}
    try:
        link_handle = Entrez.elink(dbfrom="pubmed", db="pubmed", linkname="pubmed_pubmed_citedin", id=",".join(pmids))
        link_records = Entrez.read(link_handle)
        link_handle.close()
        for record in link_records:
            if record.get('IdList'):
                source_id = record['IdList'][0]
                count = 0
                for linksetdb in record.get('LinkSetDb', []):
                    if linksetdb.get('LinkName') == 'pubmed_pubmed_citedin':
                        count = len(linksetdb.get('Link', []))
                citation_counts[source_id] = count
    except Exception as e:
        print(f"Warning: Could not fetch citation counts: {e}")

    paper_data = []

    # Iterate through the returned articles
    for record in records.get('PubmedArticle', []):
        article = record['MedlineCitation']['Article']
        pmid = str(record['MedlineCitation']['PMID'])

        # Extract Title
        title = article.get('ArticleTitle', 'No title available')

        # Extract Abstract
        abstract_text = "No abstract available"
        if 'Abstract' in article and 'AbstractText' in article['Abstract']:
            # AbstractText is returned as a list; join it to handle structured abstracts
            abstract_text = " ".join(article['Abstract']['AbstractText'])

        # Extract Date
        pub_date = article['Journal']['JournalIssue']['PubDate']
        year = pub_date.get('Year', '')
        month = pub_date.get('Month', '')
        day = pub_date.get('Day', '')
        formatted_date = f"{year} {month} {day}".strip()

        # Extract PMCID and Full Text
        pmcid = None
        for article_id in record.get('PubmedData', {}).get('ArticleIdList', []):
            if article_id.attributes.get('IdType') == 'pmc':
                pmcid = str(article_id)

        full_text = "No full text available (Not found in PubMed Central Open Access)."
        if pmcid:
            try:
                pmc_handle = Entrez.efetch(db="pmc", id=pmcid, retmode="xml")
                pmc_xml = pmc_handle.read()
                pmc_handle.close()

                pmc_root = ET.fromstring(pmc_xml)
                paragraphs = pmc_root.findall('.//body//p')
                if paragraphs:
                    full_text = "\n".join([p.text for p in paragraphs if p.text])
                else:
                    full_text = "PMCID found, but no parseable text body."
            except Exception as e:
                full_text = f"Error fetching full text: {e}"

        paper_data.append({
            'pmid': pmid,
            'title': title,
            'date': formatted_date,
            'abstract': abstract_text,
            'pmcid': pmcid,
            'full_text': full_text,
            'citations': citation_counts.get(pmid, "N/A")
        })

    return paper_data

# Pass a list of string PMIDs
pmids_to_lookup = ['26304238', '28087314']
results = get_pubmed_metadata(pmids_to_lookup)

for paper in results:
    print(f"Title: {paper['title']}")
    print(f"Date: {paper['date']}")
    print(f"Citations: {paper['citations']}")
    print(f"Abstract: {paper['abstract'][:150]}...\n")
    print(f"Full Text Snippet: {paper['full_text'][:300]}...\n")
    print("-" * 80)


In [ ]:
import math
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Citation-Based Weighting (Impact)
def calculate_citation_weight(citations, normalize=False, years_since_pub=1):
    """Calculates weight based on citations, optionally normalizing by paper age."""
    if citations == 'N/A' or citations is None:
        return 0.0
    cites = float(citations)
    if normalize and years_since_pub > 0:
        return cites / years_since_pub
    return cites

# Temporal Decay Weighting (Novelty)
def calculate_temporal_weight(pub_date_str, decay_rate=0.1):
    """Calculates exponential decay weight based on publication year."""
    try:
        # Extract year from 'YYYY Mon' format (e.g., '2016 Jul')
        pub_year = int(pub_date_str.split()[0])
        current_year = datetime.now().year
        age = max(0, current_year - pub_year)
        # Exponential decay formula: w = e^(-lambda * t)
        return math.exp(-decay_rate * age)
    except (ValueError, AttributeError, IndexError):
        # Fallback weight if date is unparseable
        return 0.5

# Semantic Relevance Weighting (Contextual)
def calculate_semantic_weight(abstract_text, target_query):
    """Calculates TF-IDF cosine similarity between an abstract and a research query."""
    if abstract_text == "No abstract available" or not abstract_text:
        return 0.0

    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        # Vectorize both the query and the abstract
        tfidf_matrix = vectorizer.fit_transform([target_query, abstract_text])
        # Compute similarity between the two vectors
        sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return float(sim)
    except ValueError:
        return 0.0

# --- Testing the Baselines on our Fetched Data ---
# Let's define a dummy research interest to test the semantic weighting
target_research_query = "caffeine effects on heart development and cardiomyocytes"

print("--- Baseline Weighting Results ---")
for paper in results:
    print(f"PMID: {paper['pmid']} | Title: {paper['title'][:60]}...")

    # Calculate age for citation normalization
    try:
        pub_year = int(paper['date'].split()[0])
        age = max(1, datetime.now().year - pub_year)
    except:
        age = 1

    w_cite = calculate_citation_weight(paper['citations'], normalize=True, years_since_pub=age)
    w_temp = calculate_temporal_weight(paper['date'], decay_rate=0.1)
    w_sem = calculate_semantic_weight(paper['abstract'], target_research_query)

    print(f"  Citation Weight (Cites/Year): {w_cite:.2f}")
    print(f"  Temporal Weight (Decay=0.1):  {w_temp:.2f}")
    print(f"  Semantic Weight (Similarity): {w_sem:.2f}\n")

In [ ]:
!pip install sentence-transformers

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util

# Check if CUDA is available and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load a domain-specific pre-trained sentence transformer model
embedding_model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device=device)

def calculate_cuda_semantic_weight(abstract_text, target_query, model):
    """Calculates semantic similarity using dense neural embeddings on CUDA."""
    if abstract_text == "No abstract available" or not abstract_text:
        return 0.0

    # Encode the query and the abstract to get their embeddings (tensors on the GPU)
    query_embedding = model.encode(target_query, convert_to_tensor=True)
    abstract_embedding = model.encode(abstract_text, convert_to_tensor=True)

    # Compute cosine similarity between the embeddings
    cosine_scores = util.cos_sim(query_embedding, abstract_embedding)

    # Extract the float value from the tensor
    return cosine_scores[0][0].item()

# --- Testing the CUDA Semantic Baseline ---
print("\n--- CUDA Semantic Weighting Results (PubMedBERT) ---")
for paper in results:
    print(f"PMID: {paper['pmid']} | Title: {paper['title'][:60]}...")

    w_sem_cuda = calculate_cuda_semantic_weight(paper['abstract'], target_research_query, embedding_model)
    print(f"  PubMedBERT Semantic Weight (Similarity): {w_sem_cuda:.2f}\n")

### Simulating Composite Weights
Combining metrics using:
1. **Paper Level:** Multiplicative (Geometric)
2. **Edge Level:** Logarithmic Scaling

In [ ]:
import math

def calculate_composite_paper_weight(w_cite, w_temp, w_sem):
    """
    Step 1: Multiplicative compositing for a single paper.
    We add a small epsilon to avoid absolute zeroing if one metric is strictly 0.
    """
    epsilon = 0.01
    return (w_cite + epsilon) * (w_temp + epsilon) * (w_sem + epsilon)

def calculate_edge_weight(paper_weights):
    """
    Step 2: Logarithmic aggregation for the edge.
    """
    total_sum = sum(paper_weights)
    return math.log1p(total_sum) # math.log1p(x) is log(1 + x)

# --- Simulation ---
print("--- Composite Weight Simulation ---")
paper_composite_scores = []

for paper in results:
    # Recalculate baselines (using CUDA for semantic if you prefer, or basic TF-IDF)
    try:
        pub_year = int(paper['date'].split()[0])
        age = max(1, datetime.now().year - pub_year)
    except:
        age = 1

    w_cite = calculate_citation_weight(paper['citations'], normalize=True, years_since_pub=age)
    w_temp = calculate_temporal_weight(paper['date'], decay_rate=0.1)
    # Let's use the CUDA PubMedBERT semantic weight we generated earlier
    w_sem = calculate_cuda_semantic_weight(paper['abstract'], target_research_query, embedding_model)

    # Step 1: Combine into one score per paper
    composite_score = calculate_composite_paper_weight(w_cite, w_temp, w_sem)
    paper_composite_scores.append(composite_score)

    print(f"PMID {paper['pmid']} Composite Score: {composite_score:.4f}")

# Step 2: Combine all paper scores for this edge
final_edge_weight = calculate_edge_weight(paper_composite_scores)
print(f"\nFinal Edge Weight (representing the relationship): {final_edge_weight:.4f}")


#Bortezomib Test

In [ ]:
import networkx as nx
from neo4j import GraphDatabase
from collections import deque

# Initialize an empty NetworkX graph
G = nx.Graph()

print("Starting BFS to construct subgraph centered on Bortezomib...")
try:
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        with driver.session() as session:
            # 1. Find the starting node (Bortezomib)
            start_query = """
            MATCH (n)
            WHERE toLower(n.name) = 'bortezomib'
            RETURN n
            """
            result = session.run(start_query)
            start_node = None
            for record in result:
                start_node = record["n"]
                break

            if start_node:
                start_id = start_node.element_id
                G.add_node(start_id, **dict(start_node.items()), labels=list(start_node.labels))

                # BFS Queue: stores (node_element_id, current_hop_depth)
                queue = deque([(start_id, 0)])
                visited = {start_id}
                max_hops = 3  # Hop limit for BFS
                max_nodes = 50000 # Safety limit to prevent memory exhaustion

                print("Traversing relationships...")
                while queue and len(visited) < max_nodes:
                    current_id, depth = queue.popleft()

                    if depth >= max_hops:
                        continue

                    # Query relationships for the current node (removed the strict PubMedIDs filter here)
                    bfs_query = """
                    MATCH (n)-[r]-(m)
                    WHERE elementId(n) = $node_id
                    RETURN n, r, m
                    """

                    neighbors_result = session.run(bfs_query, node_id=current_id)

                    for record in neighbors_result:
                        n_n = record["n"]
                        rel = record["r"]
                        m_n = record["m"]

                        m_id = m_n.element_id

                        # Add nodes and edge to graph
                        if m_id not in G:
                            G.add_node(m_id, **dict(m_n.items()), labels=list(m_n.labels))

                        G.add_edge(current_id, m_id, type=rel.type, **dict(rel.items()))

                        # Queue the neighbor if not visited
                        if m_id not in visited:
                            visited.add(m_id)
                            queue.append((m_id, depth + 1))

                print(f"\nNetworkX Graph successfully constructed via BFS!")
                print(f"Total Nodes: {G.number_of_nodes()}")
                print(f"Total Edges: {G.number_of_edges()}")
            else:
                print("Bortezomib node not found in the database.")

except Exception as e:
    print(f"Error fetching data or building graph: {e}")


In [ ]:
unique_pmids = set()

print("Scanning edges for PubMedIDs...")
# Iterate through all edges and their properties in the NetworkX graph
for u, v, data in G.edges(data=True):
    if 'PubMedIDs' in data and data['PubMedIDs'] is not None:
        val = data['PubMedIDs']

        # Handle if it's already a list
        if isinstance(val, list):
            for pmid in val:
                unique_pmids.add(str(pmid).strip())

        # Handle if it's a string (potentially comma or semicolon separated)
        elif isinstance(val, str):
            # Standardize delimiters and split
            parts = val.replace(';', ',').split(',')
            for pmid in parts:
                clean_pmid = pmid.strip()
                if clean_pmid:
                    unique_pmids.add(clean_pmid)

        # Fallback for unexpected single numeric or other types
        else:
            unique_pmids.add(str(val).strip())

# Convert set to a list for easier batching later
unique_pmid_list = list(unique_pmids)

print(f"Extraction complete!")
print(f"Found {len(unique_pmid_list)} unique PubMedIDs across the {G.number_of_edges()} edges.")
if len(unique_pmid_list) > 0:
    print(f"Sample PMIDs: {unique_pmid_list[:10]}")


In [ ]:
import time

print(f"Fetching metadata for {len(unique_pmid_list)} papers in batches...")

bortezomib_paper_metadata = []
batch_size = 500

for i in range(0, len(unique_pmid_list), batch_size):
    batch_pmids = unique_pmid_list[i:i + batch_size]
    print(f"Fetching batch {i // batch_size + 1} ({len(batch_pmids)} PMIDs)...")
    try:
        batch_metadata = get_pubmed_metadata(batch_pmids)
        bortezomib_paper_metadata.extend(batch_metadata)
        time.sleep(1) # Add a small delay to respect API limits
    except Exception as e:
        print(f"Error fetching batch {i // batch_size + 1}: {e}")

print(f"\nSuccessfully fetched metadata for {len(bortezomib_paper_metadata)} papers!")

# Let's preview the first 3 results to ensure everything looks correct
print("\n--- Sample Fetched Metadata ---")
for paper in bortezomib_paper_metadata[:3]:
    print(f"PMID: {paper['pmid']}")
    print(f"Title: {paper['title']}")
    print(f"Date: {paper['date']}")
    print(f"Citations: {paper['citations']}")
    print("-" * 60)


In [ ]:
import ast

print("Parsing stringified lists back into Python lists for G...")
list_attributes = ['PubMedIDs', 'sources', 'actions', 'labels']

parsed_edges = 0
for u, v, data in G.edges(data=True):
    for attr in list_attributes:
        if attr in data and isinstance(data[attr], str):
            val = data[attr].strip()
            try:
                # Parse string representations of lists like "['123', '456']"
                if val.startswith('[') and val.endswith(']'):
                    data[attr] = ast.literal_eval(val)
                    if attr == 'PubMedIDs':
                        parsed_edges += 1
                # Fallback for comma-separated strings
                else:
                    data[attr] = [item.strip() for item in val.split(',') if item.strip()]
                    if attr == 'PubMedIDs':
                        parsed_edges += 1
            except (ValueError, SyntaxError):
                pass # If parsing fails, keep it as a string

for node, data in G.nodes(data=True):
    for attr in list_attributes:
        if attr in data and isinstance(data[attr], str):
            val = data[attr].strip()
            try:
                if val.startswith('[') and val.endswith(']'):
                    data[attr] = ast.literal_eval(val)
                else:
                    data[attr] = [item.strip() for item in val.split(',') if item.strip()]
            except (ValueError, SyntaxError):
                pass

print(f"Successfully parsed attributes in G! (Parsed PubMedIDs in {parsed_edges} edges)")

In [ ]:
import math
from datetime import datetime
import torch
from sentence_transformers import SentenceTransformer, util

# Redefine necessary functions and models in case they are not in memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device=device)

def calculate_cuda_semantic_weight(abstract_text, target_query, model):
    if abstract_text == "No abstract available" or not abstract_text:
        return 0.0
    query_embedding = model.encode(target_query, convert_to_tensor=True)
    abstract_embedding = model.encode(abstract_text, convert_to_tensor=True)
    cosine_scores = util.cos_sim(query_embedding, abstract_embedding)
    # Clip to 0 to prevent negative weights causing math domain errors in downstream log aggregations
    return max(0.0, cosine_scores[0][0].item())

def calculate_citation_weight(citations, normalize=False, years_since_pub=1):
    if citations == 'N/A' or citations is None:
        return 0.0
    cites = float(citations)
    if normalize and years_since_pub > 0:
        return cites / years_since_pub
    return cites

def calculate_temporal_weight(pub_date_str, decay_rate=0.1):
    try:
        pub_year = int(pub_date_str.split()[0])
        current_year = datetime.now().year
        age = max(0, current_year - pub_year)
        return math.exp(-decay_rate * age)
    except:
        return 0.5

def calculate_composite_paper_weight(w_cite, w_temp, w_sem):
    epsilon = 0.01
    return (w_cite + epsilon) * (w_temp + epsilon) * (w_sem + epsilon)

def calculate_edge_weight(paper_weights):
    total_sum = sum(paper_weights)
    return math.log1p(total_sum)

# Create a dictionary for O(1) lookup of paper metadata by PMID
metadata_dict = {paper['pmid']: paper for paper in bortezomib_paper_metadata}

edges_weighted = 0
sample_queries_shown = 0

print("Calculating dynamic edge weights based on node relationships...")

# Iterate through all edges in the NetworkX graph
for u, v, data in G.edges(data=True):
    # Skip edges that already have a non-zero weight
    if data.get('weight', 0.0) != 0.0:
        continue

    if 'PubMedIDs' in data and data['PubMedIDs'] is not None:
        # Extract PMIDs
        val = data['PubMedIDs']
        pmids = []
        if isinstance(val, list):
            pmids = [str(p).strip() for p in val]
        elif isinstance(val, str):
            pmids = [p.strip() for p in val.replace(';', ',').split(',') if p.strip()]
        else:
            pmids = [str(val).strip()]

        # Construct Dynamic Target Query
        start_name = G.nodes[u].get('name', 'Unknown')
        end_name = G.nodes[v].get('name', 'Unknown')

        dynamic_query = f"{start_name} {end_name}"

        paper_scores = []

        # Calculate weights for each paper supporting this edge
        for pmid in pmids:
            if pmid in metadata_dict:
                paper = metadata_dict[pmid]

                try:
                    pub_year = int(paper['date'].split()[0])
                    age = max(1, datetime.now().year - pub_year)
                except:
                    age = 1

                w_cite = calculate_citation_weight(paper['citations'], normalize=True, years_since_pub=age)
                w_temp = calculate_temporal_weight(paper['date'], decay_rate=0.1)
                w_sem = calculate_cuda_semantic_weight(paper['abstract'], dynamic_query, embedding_model)

                composite_score = calculate_composite_paper_weight(w_cite, w_temp, w_sem)
                paper_scores.append(composite_score)

        # Aggregate paper scores into final edge weight
        if paper_scores:
            final_weight = calculate_edge_weight(paper_scores)
            G[u][v]['weight'] = final_weight
            edges_weighted += 1

            if sample_queries_shown < 5:
                print(f"\nEdge: {start_name} -> {end_name}")
                print(f"  Dynamic Query: '{dynamic_query}'")
                print(f"  Papers evaluated: {len(paper_scores)}")
                print(f"  Final Edge Weight: {final_weight:.4f}")
                sample_queries_shown += 1
        else:
            G[u][v]['weight'] = 0.0

print(f"\nSuccess! Calculated and assigned dynamic weights to {edges_weighted} edges in the NetworkX graph.")

In [ ]:
# Locate the specific node ID for Bortezomib in our NetworkX graph
bortezomib_id = None
for node_id, data in G.nodes(data=True):
    if data.get('name', '').lower() == 'bortezomib':
        bortezomib_id = node_id
        break

print(f"Target Node ID: {bortezomib_id}")

# Find ALL immediate neighbors and inspect their labels
if bortezomib_id:
    all_neighbors = []
    label_counts = {}

    for neighbor_id in G.neighbors(bortezomib_id):
        neighbor_data = G.nodes[neighbor_id]
        labels = tuple(neighbor_data.get('labels', [])) # Use tuple so we can count them
        name = neighbor_data.get('name', 'Unknown')

        all_neighbors.append({'name': name, 'labels': list(labels)})

        if labels in label_counts:
            label_counts[labels] += 1
        else:
            label_counts[labels] = 1

    print(f"\nFound {len(all_neighbors)} TOTAL direct connections to Bortezomib in this subgraph.")

    print("\n--- Breakdown of Neighbor Labels ---")
    for labels, count in label_counts.items():
        print(f"Labels {list(labels)}: {count} nodes")

    print("\n--- Sample of Immediate Neighbors ---")
    for item in all_neighbors[:15]:
        print(f"- {item['name']} (Labels: {item['labels']})")
else:
    print("Bortezomib node not found in the current subgraph.")

In [ ]:
# Secrewed up adding pubmedids to some relationships, this fixes that
from neo4j import GraphDatabase

# Update the in-memory NetworkX graph (G)
print("Standardizing NetworkX graph (G)...")
nx_updated = 0
for u, v, data in G.edges(data=True):
    if 'pubmed_ids' in data:
        # Transfer the value to 'PubMedIDs' and delete 'pubmed_ids'
        data['PubMedIDs'] = data.pop('pubmed_ids')
        nx_updated += 1

print(f"Updated {nx_updated} edges in NetworkX graph G.")

# Also update loaded_G just in case you use it going forward
if 'loaded_G' in locals():
    loaded_nx_updated = 0
    for u, v, data in loaded_G.edges(data=True):
        if 'pubmed_ids' in data:
            data['PubMedIDs'] = data.pop('pubmed_ids')
            loaded_nx_updated += 1
    print(f"Updated {loaded_nx_updated} edges in loaded_G.")

# Update the Neo4j Database
print("\nStandardizing Neo4j Database...")
# We use a batched approach to avoid overloading the transaction memory
update_query = """
MATCH ()-[r]->()
WHERE r.pubmed_ids IS NOT NULL
WITH r LIMIT 50000
SET r.PubMedIDs = r.pubmed_ids
REMOVE r.pubmed_ids
RETURN count(r) as updated_count
"""

try:
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        with driver.session() as session:
            total_updated = 0
            while True:
                result = session.run(update_query)
                count = result.single()["updated_count"]
                if count == 0:
                    break
                total_updated += count
                print(f"Updated {count} relationships in this batch. Total so far: {total_updated}")

            print(f"\nSuccessfully standardized {total_updated} relationships in Neo4j!")
except Exception as e:
    print(f"Could not update Neo4j (you might have read-only access or another error occurred): {e}")


In [ ]:
import networkx as nx

G = loaded_G

# We will search for variants of the target disease
search_terms = ['hematologic cancer', 'hematological cancer', 'multiple myeloma', 'lymphoma']
found_nodes = []

print("Searching the subgraph for the requested disease nodes...")

for node_id, data in G.nodes(data=True):
    name = data.get('name', '').lower()
    # Check if any of our search terms are in the node's name
    if any(term in name for term in search_terms):
        found_nodes.append((node_id, data))

if found_nodes:
    print(f"\nFound {len(found_nodes)} node(s) matching your criteria:")
    for n_id, data in found_nodes:
        print(f"\n- Node: {data.get('name')} | Labels: {data.get('labels')}")

        # Trace the path back to Bortezomib
        if bortezomib_id:
            try:
                path = nx.shortest_path(G, source=bortezomib_id, target=n_id)
                hops = len(path) - 1
                print(f"  -> Shortest path from Bortezomib is {hops} hop(s).")

                # Print the names of the nodes in the path
                path_names = [G.nodes[p].get('name', 'Unknown') for p in path]
                print(f"  -> Path: {' -> '.join(path_names)}")
            except nx.NetworkXNoPath:
                print("  -> No path found between Bortezomib and this node in the current subgraph.")
else:
    print(f"\nCould not find any nodes matching '{search_terms}' in this specific subgraph.")
    print("This suggests the specific node was not captured within the 25,000-path query limit, or the database uses a different ontology term.")


In [ ]:
import networkx as nx
import os

# Specify the save location (directory) and filename
save_directory = "/content/drive/MyDrive/"  # Change this to your desired save location
file_name = "bortezomib_subgraph.graphml"
output_path = os.path.join(save_directory, file_name)

# Ensure the directory exists
os.makedirs(save_directory, exist_ok=True)

# GraphML does not support list attributes. We need to convert lists to strings.
print("Converting list attributes to strings for GraphML compatibility...")

for node, data in G.nodes(data=True):
    for key, value in data.items():
        if isinstance(value, (list, tuple, set)):
            G.nodes[node][key] = ", ".join(map(str, value))

for u, v, data in G.edges(data=True):
    for key, value in data.items():
        if isinstance(value, (list, tuple, set)):
            G.edges[u, v][key] = ", ".join(map(str, value))

# Save the graph
print(f"Saving the graph to {output_path}...")
nx.write_graphml(G, output_path)
print(f"Graph successfully saved at: {output_path}")


In [ ]:
import networkx as nx
import os
import ast

# The path where the graph was saved
load_path = "/content/drive/MyDrive/bortezomib_subgraph.graphml"

print(f"Attempting to load graph from {load_path}...")

try:
    # Load the graphml file into a new NetworkX graph object
    loaded_G = nx.read_graphml(load_path)
    print("Graph successfully loaded!")

    list_attributes = ['pubmed_ids', 'PubMedIDs', 'sources', 'actions', 'labels']

    print("Parsing stringified lists back into Python lists...")
    for u, v, data in loaded_G.edges(data=True):
        for attr in list_attributes:
            if attr in data and isinstance(data[attr], str):
                try:
                    # Parse string representations of lists like "['123', '456']"
                    if data[attr].strip().startswith('[') and data[attr].strip().endswith(']'):
                        data[attr] = ast.literal_eval(data[attr])
                    # Fallback for comma-separated strings
                    else:
                        data[attr] = [item.strip() for item in data[attr].split(',') if item.strip()]
                except (ValueError, SyntaxError):
                    pass # If parsing fails, keep it as a string

    for node, data in loaded_G.nodes(data=True):
        for attr in list_attributes:
            if attr in data and isinstance(data[attr], str):
                try:
                    if data[attr].strip().startswith('[') and data[attr].strip().endswith(']'):
                        data[attr] = ast.literal_eval(data[attr])
                    else:
                        data[attr] = [item.strip() for item in data[attr].split(',') if item.strip()]
                except (ValueError, SyntaxError):
                    pass

    print(f"Total Nodes in loaded graph: {loaded_G.number_of_nodes()}")
    print(f"Total Edges in loaded graph: {loaded_G.number_of_edges()}")

    # Check a node and edge to ensure attributes are parsed
    sample_node = list(loaded_G.nodes(data=True))[0]
    sample_edge = list(loaded_G.edges(data=True))[0]
    print(f"\nSample loaded node: {sample_node}")
    print(f"Sample loaded edge: {sample_edge}")

except FileNotFoundError:
    print(f"Error: The file {load_path} was not found. Please ensure it was saved correctly.")
except Exception as e:
    print(f"An error occurred while loading: {e}")

# Walks

In [ ]:
import networkx as nx

# Fallback to G if loaded_G is not available for some reason
graph_to_use = loaded_G if 'loaded_G' in locals() else G

# Ensure we have the correct Bortezomib ID from the current graph
target_id = None
for node_id, data in graph_to_use.nodes(data=True):
    if data.get('name', '').lower() == 'bortezomib':
        target_id = node_id
        break

baseline_nodes = []

if target_id:
    print(f"Scanning neighbors of Bortezomib (ID: {target_id})...")
    for neighbor_id in graph_to_use.neighbors(target_id):
        neighbor_data = graph_to_use.nodes[neighbor_id]
        labels = neighbor_data.get('labels', [])

        # Check if it's a Disease or Side_Effect
        if 'Disease' in labels or 'Side_Effect' in labels:
            baseline_nodes.append(neighbor_data)

    print(f"\nFound {len(baseline_nodes)} Disease and Side_Effect nodes directly adjacent to Bortezomib.")

    print("\n--- Sample Baseline Nodes ---")
    for data in baseline_nodes[:20]:
        print(f"- {data.get('name', 'Unknown')} (Labels: {data.get('labels', [])})")
else:
    print("Bortezomib node not found in the graph.")


In [ ]:
import networkx as nx

print("Preparing edge weights for the walk...")
# Ensure graph_to_use is defined (fallback to G if needed)
graph_to_use = loaded_G if 'loaded_G' in locals() else G

# Set up a 'walk_weight' to ensure no zero-weight edges trap the walk,
# but heavily favor the dynamic weights we calculated.
for u, v, d in graph_to_use.edges(data=True):
    if 'weight' in d and d['weight'] > 0.0:
        d['walk_weight'] = d['weight']
    else:
        d['walk_weight'] = 0.000 # Small baseline probability for unweighted edges

print(f"Running Personalized PageRank (Weighted Walk) centered on Bortezomib...")
# Run Personalized PageRank
ppr_scores = nx.pagerank(graph_to_use, personalization={target_id: 1.0}, weight='walk_weight')

# Filter to only Disease and Side_Effect nodes, excluding Bortezomib itself
target_labels = {'Disease', 'Side_Effect'}
walk_results = []
for n_id, score in ppr_scores.items():
    if n_id == target_id:
        continue
    data = graph_to_use.nodes[n_id]
    labels = set(data.get('labels', []))
    if labels.intersection(target_labels):
        walk_results.append((n_id, data.get('name', 'Unknown'), score, list(labels)))

# Sort by score descending
walk_results.sort(key=lambda x: x[2], reverse=True)
top_10 = walk_results[:10]

# Create a set of lowercased baseline names for robust matching
baseline_names = {node.get('name', '').lower() for node in baseline_nodes}

print("\n--- Top 10 Disease/Side_Effect Nodes from Weighted Walk ---")
for i, (n_id, name, score, labels) in enumerate(top_10, 1):
    name_lower = name.lower()
    in_baseline = name_lower in baseline_names
    status = "IN BASELINE (Direct Connection)" if in_baseline else "NEW (Discovered via Walk)"
    print(f"{i}. {name}")
    print(f"   Score: {score:.6f} | Labels: {labels}")
    print(f"   Comparison: {status}")

    try:
        path = nx.shortest_path(graph_to_use, source=target_id, target=n_id)
        path_names = [graph_to_use.nodes[p].get('name', 'Unknown') for p in path]
        print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
    except nx.NetworkXNoPath:
        print(f"   Shortest Path: No path found.\n")


## Look-Ahead

In [ ]:
import networkx as nx

print("Calculating total edge weights for each node...")
node_weights = {}
for n in graph_to_use.nodes():
    total_weight = sum(d.get('walk_weight', 0.000) for _, _, d in graph_to_use.edges(n, data=True))
    node_weights[n] = total_weight

print("Building a directed graph to allow asymmetric transition weights...")
# We use a DiGraph so the walk favors moving towards heavier nodes
walk_graph = nx.DiGraph()
for u, v, d in graph_to_use.edges(data=True):
    base_weight = d.get('walk_weight', 0.000)
    walk_graph.add_edge(u, v, weight=base_weight * node_weights[v])
    walk_graph.add_edge(v, u, weight=base_weight * node_weights[u])

print("Running Node-Biased Personalized PageRank...")
biased_ppr_scores = nx.pagerank(walk_graph, personalization={target_id: 1.0}, weight='weight')

biased_walk_results = []
for n_id, score in biased_ppr_scores.items():
    if n_id == target_id:
        continue
    data = graph_to_use.nodes[n_id]
    labels = set(data.get('labels', []))
    if labels.intersection(target_labels):
        biased_walk_results.append((n_id, data.get('name', 'Unknown'), score, list(labels)))

biased_walk_results.sort(key=lambda x: x[2], reverse=True)
biased_top_10 = biased_walk_results[:10]

print("\n--- Top 10 Disease/Side_Effect Nodes from Node-Biased Walk ---")
for i, (n_id, name, score, labels) in enumerate(biased_top_10, 1):
    name_lower = name.lower()
    status = "IN BASELINE (Direct Connection)" if name_lower in baseline_names else "NEW (Discovered via Walk)"
    print(f"{i}. {name}")
    print(f"   Score: {score:.6f} | Labels: {labels}")
    print(f"   Comparison: {status}")

    try:
        path = nx.shortest_path(graph_to_use, source=target_id, target=n_id)
        path_names = [graph_to_use.nodes[p].get('name', 'Unknown') for p in path]
        print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
    except nx.NetworkXNoPath:
        print(f"   Shortest Path: No path found.\n")


In [ ]:
import random
from collections import Counter
import pandas as pd
from IPython.display import display
import networkx as nx

# Ensure graph_to_use is defined as before
graph_to_use = loaded_G if 'loaded_G' in locals() else G

def run_random_walks(G, start_node, num_walks=5000, walk_length=5, use_weights=True):
    visits = Counter()
    for _ in range(num_walks):
        curr = start_node
        for _ in range(walk_length):
            neighbors = list(G.neighbors(curr))
            if not neighbors:
                break

            if use_weights:
                # Use walk_weight or weight, defaulting to a small probability for unweighted edges
                weights = [max(0.0, G[curr][nbr].get('walk_weight', G[curr][nbr].get('weight', 0.001))) for nbr in neighbors]
                curr = random.choices(neighbors, weights=weights, k=1)[0]
            else:
                curr = random.choice(neighbors)

            # Track visits to Disease and Side_Effect nodes
            curr_labels = G.nodes[curr].get('labels', [])
            if ('Disease' in curr_labels or 'Side_Effect' in curr_labels) and curr != start_node:
                visits[curr] += 1

    return visits

print(f"Running {5000} random walks anchored at Bortezomib (ID: {target_id})...")
visits = run_random_walks(graph_to_use, target_id, num_walks=5000, walk_length=5, use_weights=True)

# Get direct neighbors to filter out 1-hop paths
direct_neighbors = set(graph_to_use.neighbors(target_id))

# Rank all visited diseases/side effects, excluding direct neighbors
ranked_results = []
for node_id, count in visits.items():
    # Skip 1-hop connections
    if node_id in direct_neighbors:
        continue

    data = graph_to_use.nodes[node_id]
    name = data.get('name', 'Unknown')
    labels = ", ".join(data.get('labels', []))
    ranked_results.append({
        'Node ID': node_id,
        'Disease/Side Effect': name,
        'Labels': labels,
        'Visit Count': count
    })

if ranked_results:
    # Sort by visit count descending
    ranked_results.sort(key=lambda x: x['Visit Count'], reverse=True)
    top_10 = ranked_results[:10]

    print("\n--- Top 10 Indirect (2+ Hops) Disease/Side_Effect Nodes from Random Walks ---")
    for i, res in enumerate(top_10, 1):
        n_id = res['Node ID']
        name = res['Disease/Side Effect']
        count = res['Visit Count']
        labels = res['Labels']

        print(f"{i}. {name}")
        print(f"   Visit Count: {count} | Labels: [{labels}]")
        print(f"   Comparison: NEW (Discovered via Walk)")

        try:
            path = nx.shortest_path(graph_to_use, source=target_id, target=n_id)
            path_names = [graph_to_use.nodes[p].get('name', 'Unknown') for p in path]
            print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
        except nx.NetworkXNoPath:
            print(f"   Shortest Path: No path found.\n")
else:
    print("No indirect Diseases or Side Effects were visited during the walks.")


In [ ]:
import random
from collections import Counter
import pandas as pd
from IPython.display import display
import networkx as nx

# The original graph contains all our node attributes (names, labels)
original_graph = loaded_G if 'loaded_G' in locals() else G

def run_biased_random_walks(di_G, orig_G, start_node, num_walks=5000, walk_length=5):
    visits = Counter()
    for _ in range(num_walks):
        curr = start_node
        for _ in range(walk_length):
            # Use successors since walk_graph is a DiGraph
            neighbors = list(di_G.successors(curr))
            if not neighbors:
                break

            # Extract the look-ahead weight
            weights = [max(0.0, di_G[curr][nbr].get('weight', 0.001)) for nbr in neighbors]

            if sum(weights) > 0:
                curr = random.choices(neighbors, weights=weights, k=1)[0]
            else:
                curr = random.choice(neighbors)

            # Track visits to Disease and Side_Effect nodes using original graph attributes
            curr_labels = orig_G.nodes[curr].get('labels', [])
            if ('Disease' in curr_labels or 'Side_Effect' in curr_labels) and curr != start_node:
                visits[curr] += 1

    return visits

print(f"Running {5000} look-ahead (node-biased) random walks anchored at Bortezomib (ID: {target_id})...")
visits = run_biased_random_walks(walk_graph, original_graph, target_id, num_walks=200000, walk_length=5)

# Get direct neighbors from the original graph to filter out 1-hop paths
direct_neighbors = set(original_graph.neighbors(target_id))

# Rank all visited diseases/side effects, excluding direct neighbors
ranked_results = []
for node_id, count in visits.items():
    # Skip 1-hop connections
    if node_id in direct_neighbors:
        continue

    data = original_graph.nodes[node_id]
    name = data.get('name', 'Unknown')
    labels = ", ".join(data.get('labels', []))
    ranked_results.append({
        'Node ID': node_id,
        'Disease/Side Effect': name,
        'Labels': labels,
        'Visit Count': count
    })

if ranked_results:
    # Sort by visit count descending
    ranked_results.sort(key=lambda x: x['Visit Count'], reverse=True)
    top_10 = ranked_results[:10]

    print("\n--- Top 10 Indirect (2+ Hops) Disease/Side_Effect Nodes from Biased Random Walks ---")
    for i, res in enumerate(top_10, 1):
        n_id = res['Node ID']
        name = res['Disease/Side Effect']
        count = res['Visit Count']
        labels = res['Labels']

        print(f"{i}. {name}")
        print(f"   Visit Count: {count} | Labels: [{labels}]")
        print(f"   Comparison: NEW (Discovered via Walk)")

        try:
            path = nx.shortest_path(original_graph, source=target_id, target=n_id)
            path_names = [original_graph.nodes[p].get('name', 'Unknown') for p in path]
            print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
        except nx.NetworkXNoPath:
            print(f"   Shortest Path: No path found.\n")
else:
    print("No indirect Diseases or Side Effects were visited during the biased walks.")


## Weight Limited

In [ ]:
import random
from collections import Counter
import networkx as nx

def run_capacity_random_walks(di_G, orig_G, start_node, num_walks=5000, max_hops=5, max_total_weight=1.0):
    visits = Counter()
    for _ in range(num_walks):
        curr = start_node
        path_weight = 0.0

        for _ in range(max_hops):
            # Use successors since walk_graph is a DiGraph
            neighbors = list(di_G.successors(curr))
            if not neighbors:
                break

            # Extract weights to compute probabilities
            weights = [max(0.0, di_G[curr][nbr].get('weight', 0.001)) for nbr in neighbors]

            if sum(weights) > 0:
                next_node = random.choices(neighbors, weights=weights, k=1)[0]
            else:
                next_node = random.choice(neighbors)

            # Add the current edge weight to our running total
            step_weight = di_G[curr][next_node].get('weight', 0.0)
            path_weight += step_weight

            # If we exceed the budget, terminate this walk early
            if path_weight > max_total_weight:
                break

            curr = next_node

            # Track visits to Disease and Side_Effect nodes
            curr_labels = orig_G.nodes[curr].get('labels', [])
            if ('Disease' in curr_labels or 'Side_Effect' in curr_labels) and curr != start_node:
                visits[curr] += 1

    return visits

MAX_WEIGHT_LIMIT = 0.3  # You can adjust this capacity constraint
num_walks_capacity = 500000

print(f"Running {num_walks_capacity} capacity-constrained walks anchored at Bortezomib...")
print(f"Max additive weight allowed per path: {MAX_WEIGHT_LIMIT}\n")

visits_cap = run_capacity_random_walks(walk_graph, original_graph, target_id,
                                       num_walks=num_walks_capacity,
                                       max_hops=5,
                                       max_total_weight=MAX_WEIGHT_LIMIT)

# Filter and format results
ranked_cap_results = []
for node_id, count in visits_cap.items():
    if node_id in direct_neighbors:
        continue

    data = original_graph.nodes[node_id]
    name = data.get('name', 'Unknown')
    labels = ", ".join(data.get('labels', []))
    ranked_cap_results.append({
        'Node ID': node_id,
        'Disease/Side Effect': name,
        'Labels': labels,
        'Visit Count': count
    })

if ranked_cap_results:
    ranked_cap_results.sort(key=lambda x: x['Visit Count'], reverse=True)
    top_10_cap = ranked_cap_results[:10]

    print(f"--- Top 10 Indirect Nodes (Capacity Walk | Max Weight: {MAX_WEIGHT_LIMIT}) ---")
    for i, res in enumerate(top_10_cap, 1):
        n_id = res['Node ID']
        name = res['Disease/Side Effect']
        count = res['Visit Count']

        print(f"{i}. {name}")
        print(f"   Visit Count: {count} | Labels: [{res['Labels']}]")

        try:
            path = nx.shortest_path(original_graph, source=target_id, target=n_id)
            path_names = [original_graph.nodes[p].get('name', 'Unknown') for p in path]
            print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
        except nx.NetworkXNoPath:
            print(f"   Shortest Path: No path found.\n")
else:
    print("No indirect nodes were reached within the weight capacity limit.")

In [ ]:
from collections import Counter
import random

def run_progressive_walks(G, start_node, num_walks=1000, walk_length=4):
    visits = Counter()
    for _ in range(num_walks):
        curr = start_node
        for step in range(walk_length):
            neighbors = list(G.neighbors(curr))
            if not neighbors:
                break

            alpha = step / max(1, (walk_length - 1))

            weights = [max(0.0, G[curr][nbr].get('weight', 0.01)) for nbr in neighbors]
            total_weight = sum(weights)

            p_unweighted = 1.0 / len(neighbors)

            final_probs = []
            for w in weights:
                p_weighted = w / total_weight
                p_final = (1 - alpha) * p_unweighted + alpha * p_weighted
                final_probs.append(p_final)

            curr = random.choices(neighbors, weights=final_probs, k=1)[0]

            # FIXED: Check 'labels' instead of 'type'
            if 'Disease' in G.nodes[curr].get('labels', []) and curr != start_node:
                visits[curr] += 1

    return visits

## Unweighted to Weighted

In [ ]:
import pandas as pd
import networkx as nx
from IPython.display import display

target_cmpd_name = 'Bortezomib'
print(f"Running progressive weight random walks anchored at {target_cmpd_name}...")

# 1. Find the correct NetworkX node ID for Bortezomib
target_id = None
for node_id, data in loaded_G.nodes(data=True):
    if data.get('name', '').lower() == target_cmpd_name.lower():
        target_id = node_id
        break

if target_id:
    # Recalculate the nodes within 2 to 5 hops using the correct ID
    lengths = nx.single_source_shortest_path_length(loaded_G, target_id, cutoff=5)

    # Note: We check 'labels' instead of 'type' based on the graph's schema
    bortezomib_diseases_2_to_5_hops = [
        node for node, dist in lengths.items()
        if 1 < dist <= 5 and 'Disease' in loaded_G.nodes[node].get('labels', [])
    ]

    # Run the progressive walk using the correct node ID
    visits_progressive = run_progressive_walks(loaded_G, target_id, num_walks=5000, walk_length=5)

    # Rank only the diseases that are within 2-5 hops
    ranked_results_progressive = []
    for disease_id in bortezomib_diseases_2_to_5_hops:
        count = visits_progressive.get(disease_id, 0)
        disease_name = loaded_G.nodes[disease_id].get('name', 'Unknown')
        ranked_results_progressive.append({
            'Disease': disease_name,
            'Progressive Visit Count': count
        })

    # Convert to a DataFrame and sort by visit count descending
    df_ranked_progressive = pd.DataFrame(ranked_results_progressive)
    if not df_ranked_progressive.empty:
        df_ranked_progressive = df_ranked_progressive.sort_values(by='Progressive Visit Count', ascending=False).reset_index(drop=True)

    print(f"Top 20 Ranked Diseases (Progressive, 2-5 hops) for {target_cmpd_name}:")
    display(df_ranked_progressive.head(20))
else:
    print(f"{target_cmpd_name} not found in the graph.")

## Dynamic Weight

run_dynamic_weight_random_walks (Persists across ALL walks)
In this function, the edge_weights = {} dictionary is initialized outside the num_walks loop.

What this means: The weight changes are global and persist across all 500,000 walks. It acts like a global "pheromone trail" for the whole swarm.
The Math: If a multiplier is greater than 1 (e.g., 1.2 or 2.0), the growth is exponential over thousands of crossings. If an edge is crossed just 1,000 times out of the 500,000 walks with a factor of 2.0, its weight becomes $Original \times 2^{1000}$$Original \times 2^{1000}$. This causes an immediate floating-point overflow (which is why you had to add the min(..., 1e100) safeguard).
Even with the safeguard capping it at 1e100, that edge instantly becomes a "super-highway black hole." Because random.choices uses relative weights, an edge with a weight of 1e100 makes the probability of choosing any other adjacent edge effectively $0\%$.

In [ ]:
import random
from collections import Counter
import networkx as nx

def run_dynamic_weight_random_walks(di_G, orig_G, start_node, num_walks=50000, max_hops=5, decay_factor=0.8):
    visits = Counter()

    # 1. Initialize a separate dictionary to track dynamic weights
    # This prevents us from permanently altering the main graph's attributes
    edge_weights = {}
    for u, v, d in di_G.edges(data=True):
        edge_weights[(u, v)] = max(0.0, d.get('weight', 0.001))

    for _ in range(num_walks):
        curr = start_node
        # NEW: Track visited nodes in this specific walk to prevent backtracking
        visited = {curr}

        for _ in range(max_hops):
            # Use successors since walk_graph is a DiGraph and filter out visited nodes
            neighbors = [n for n in di_G.successors(curr) if n not in visited]
            if not neighbors:
                break

            # Extract current dynamic weights from our tracking dictionary
            weights = [edge_weights[(curr, nbr)] for nbr in neighbors]

            if sum(weights) > 0:
                next_node = random.choices(neighbors, weights=weights, k=1)[0]
            else:
                next_node = random.choice(neighbors)

            # 2. Temporarily modify the weight of the crossed edge
            # Added a cap (1e100) to prevent float overflow if decay_factor > 1
            edge_weights[(curr, next_node)] = min(edge_weights[(curr, next_node)] * decay_factor, 1e2)

            curr = next_node
            visited.add(curr)

            # 3. Track visits to Disease and Side_Effect nodes
            curr_labels = orig_G.nodes[curr].get('labels', [])
            if ('Disease' in curr_labels or 'Side_Effect' in curr_labels) and curr != start_node:
                visits[curr] += 1

    return visits

DECAY_FACTOR = 1.1
num_walks_dynamic =50000

print(f"Running {num_walks_dynamic} dynamic-weight walks anchored at Bortezomib...")
print(f"Edge weight decay/growth factor per crossing: {DECAY_FACTOR}\n")

visits_dynamic = run_dynamic_weight_random_walks(walk_graph, original_graph, target_id,
                                                 num_walks=num_walks_dynamic,
                                                 max_hops=5,
                                                 decay_factor=DECAY_FACTOR)

# Filter and format results
ranked_dyn_results = []
for node_id, count in visits_dynamic.items():
    if node_id in direct_neighbors:
        continue

    data = original_graph.nodes[node_id]
    name = data.get('name', 'Unknown')
    labels = ", ".join(data.get('labels', []))
    ranked_dyn_results.append({
        'Node ID': node_id,
        'Disease/Side Effect': name,
        'Labels': labels,
        'Visit Count': count
    })

if ranked_dyn_results:
    ranked_dyn_results.sort(key=lambda x: x['Visit Count'], reverse=True)
    top_10_dyn = ranked_dyn_results[:10]

    print(f"--- Top 10 Indirect Nodes (Dynamic Weight Walk | Factor: {DECAY_FACTOR}) ---")
    for i, res in enumerate(top_10_dyn, 1):
        n_id = res['Node ID']
        name = res['Disease/Side Effect']
        count = res['Visit Count']

        print(f"{i}. {name}")
        print(f"   Visit Count: {count} | Labels: [{res['Labels']}]")

        try:
            path = nx.shortest_path(original_graph, source=target_id, target=n_id)
            path_names = [original_graph.nodes[p].get('name', 'Unknown') for p in path]
            print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
        except nx.NetworkXNoPath:
            print(f"   Shortest Path: No path found.\n")
else:
    print("No indirect nodes were reached during the dynamic weight walks.")

run_dynamic_weight_walks (Resets per walk)
In this function, the dynamic_weights = {} dictionary is initialized inside the for _ in range(num_walks): loop.

What this means: The weights reset to their original baseline for every single walker.
The Math: A walker only takes a few steps (e.g., walk_length = 5). If it crosses the same edge 5 times with a multiplier of 2.0, the maximum weight that edge can reach is $2^5 = 32$$2^5 = 32$ times its original weight. This is perfectly safe for Python's floating-point math and acts to reinforce a path only for that specific walker (like a person picking up speed as they run down a hill).

In [ ]:
from collections import Counter
import random

def run_dynamic_weight_walks(G, start_node, num_walks=1000, walk_length=4, multiplier=2.0):
    visits = Counter()
    for _ in range(num_walks):
        curr = start_node
        dynamic_weights = {}
        for _ in range(walk_length):
            neighbors = list(G.neighbors(curr))
            if not neighbors:
                break

            current_weights = []
            for nbr in neighbors:
                edge = tuple(sorted((curr, nbr)))
                if edge not in dynamic_weights:
                    dynamic_weights[edge] = max(0.0, G[curr][nbr].get('weight', 0.01))
                current_weights.append(dynamic_weights[edge])

            next_node = random.choices(neighbors, weights=current_weights, k=1)[0]

            # Update the weight of the traversed edge
            traversed_edge = tuple(sorted((curr, next_node)))
            dynamic_weights[traversed_edge] *= multiplier

            curr = next_node

            # FIXED: Check 'labels' instead of 'type' just like the other walk functions
            if 'Disease' in G.nodes[curr].get('labels', []) and curr != start_node:
                visits[curr] += 1

    return visits

In [ ]:
import pandas as pd
import networkx as nx

target_cmpd_name = 'Bortezomib'
print(f"Running dynamic weight random walks anchored at {target_cmpd_name}...")

# Find the correct NetworkX node ID for Bortezomib
target_id = None
for node_id, data in loaded_G.nodes(data=True):
    if data.get('name', '').lower() == target_cmpd_name.lower():
        target_id = node_id
        break

if target_id:
    # Run the dynamic weight walk using the correct node ID
    visits_dynamic = run_dynamic_weight_walks(loaded_G, target_id, num_walks=50000, walk_length=5, multiplier=2.0)

    # Get direct neighbors to filter out 1-hop paths
    direct_neighbors = set(loaded_G.neighbors(target_id))

    # Rank all visited diseases/side effects, excluding direct neighbors
    ranked_results_dynamic = []
    for node_id, count in visits_dynamic.items():
        if node_id in direct_neighbors:
            continue

        data = loaded_G.nodes[node_id]
        name = data.get('name', 'Unknown')
        labels = ", ".join(data.get('labels', []))
        ranked_results_dynamic.append({
            'Node ID': node_id,
            'Disease/Side Effect': name,
            'Labels': labels,
            'Visit Count': count
        })

    if ranked_results_dynamic:
        # Sort by visit count descending
        ranked_results_dynamic.sort(key=lambda x: x['Visit Count'], reverse=True)
        top_10 = ranked_results_dynamic[:10]

        print(f"\n--- Top 10 Indirect Nodes (Dynamic Weight Walk | Multiplier: 2.0) ---")
        for i, res in enumerate(top_10, 1):
            n_id = res['Node ID']
            name = res['Disease/Side Effect']
            count = res['Visit Count']

            print(f"{i}. {name}")
            print(f"   Visit Count: {count} | Labels: [{res['Labels']}]")

            try:
                path = nx.shortest_path(loaded_G, source=target_id, target=n_id)
                path_names = [loaded_G.nodes[p].get('name', 'Unknown') for p in path]
                print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
            except nx.NetworkXNoPath:
                print(f"   Shortest Path: No path found.\n")
    else:
        print("No indirect nodes were reached during the dynamic weight walks.")
else:
    print(f"{target_cmpd_name} not found in the graph.")


## Metropolis-Hastings MCMC

In [ ]:
import random
import math
from collections import Counter
import pandas as pd
from IPython.display import display
import networkx as nx

def run_mcmc_mh_walk(G, start_node, prior_scores, num_samples=10000, temperature=1.0):
    """
    Metropolis-Hastings MCMC sampling on a graph.

    Prior/Target (pi): Personalized PageRank scores (prior_scores).
    Temperature (T): Controls exploration vs. exploitation.
                     T < 1.0 sharpens the distribution (greedy).
                     T > 1.0 flattens the distribution (random).
    """
    visits = Counter()
    curr = start_node

    def get_pi(node):
        # Add a small epsilon to prevent ZeroDivision or log(0) issues
        return prior_scores.get(node, 1e-12)

    for _ in range(num_samples):
        # Proposal q(x' | x): Pick a neighbor uniformly
        neighbors = list(G.neighbors(curr))
        if not neighbors:
            break

        proposed = random.choice(neighbors)

        # Target distribution with Temperature: pi(x)^(1/T)
        pi_curr = get_pi(curr) ** (1.0 / temperature)
        pi_prop = get_pi(proposed) ** (1.0 / temperature)

        # Proposal probabilities q(curr|prop) / q(prop|curr)
        q_prop_given_curr = 1.0 / len(neighbors)
        proposed_neighbors_count = len(list(G.neighbors(proposed)))
        q_curr_given_prop = 1.0 / proposed_neighbors_count if proposed_neighbors_count > 0 else 0.0

        # Calculate Acceptance Ratio
        if pi_curr * q_prop_given_curr == 0:
            acceptance_prob = 1.0
        else:
            ratio = (pi_prop * q_curr_given_prop) / (pi_curr * q_prop_given_curr)
            acceptance_prob = min(1.0, ratio)

        # Accept or Reject
        if random.random() < acceptance_prob:
            curr = proposed

        # Track visits to Diseases/Side Effects (excluding 1-hop baseline nodes if needed later)
        curr_labels = G.nodes[curr].get('labels', [])
        if ('Disease' in curr_labels or 'Side_Effect' in curr_labels) and curr != start_node:
            visits[curr] += 1

    return visits

# --- Execution ---
TEMPERATURE = 0.9  # Try setting to 0.1 (greedy) or 5.0 (exploratory)
NUM_SAMPLES = 500000

print(f"Running MCMC Metropolis-Hastings (T={TEMPERATURE})...")

# Ensure we use the PPR scores calculated in earlier cells
mh_visits = run_mcmc_mh_walk(
    graph_to_use,
    target_id,
    ppr_scores,
    num_samples=NUM_SAMPLES,
    temperature=TEMPERATURE
)

# Filter and Rank
ranked_mh = []
for node_id, count in mh_visits.items():
    # Skip 1-hop direct neighbors to focus on novel discoveries
    if node_id in direct_neighbors:
        continue

    data = graph_to_use.nodes[node_id]
    ranked_mh.append({
        'Node ID': node_id,
        'Disease/Side Effect': data.get('name', 'Unknown'),
        'Labels': ", ".join(data.get('labels', [])),
        'Visit Count': count
    })

if ranked_mh:
    ranked_mh.sort(key=lambda x: x['Visit Count'], reverse=True)
    top_15 = ranked_mh[:15]

    print(f"\n--- Top 15 Indirect Discoveries via MCMC (Temperature: {TEMPERATURE}) ---")
    for i, res in enumerate(top_15, 1):
        n_id = res['Node ID']
        name = res['Disease/Side Effect']
        count = res['Visit Count']
        labels = res['Labels']

        print(f"{i}. {name}")
        print(f"   Visit Count: {count} | Labels: [{labels}]")

        try:
            path = nx.shortest_path(graph_to_use, source=target_id, target=n_id)
            path_names = [graph_to_use.nodes[p].get('name', 'Unknown') for p in path]
            print(f"   Shortest Path ({len(path)-1} hops): {' -> '.join(path_names)}\n")
        except nx.NetworkXNoPath:
            print(f"   Shortest Path: No path found.\n")
else:
    print("No indirect nodes visited during MCMC.")


## Combined Walk Results Visualization
This graph aggregates the visit counts from all the different walk algorithms (Random, Capacity, Dynamic, MCMC). Nodes are sized based on their total visit count across all experiments.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter

# Aggregate visit counts from all available results
total_visits = Counter()

# List of result datasets to combine
lists_to_process = [
    ('Standard', ranked_results if 'ranked_results' in locals() else []),
    ('Capacity', ranked_cap_results if 'ranked_cap_results' in locals() else []),
    ('Dynamic (Global)', ranked_dyn_results if 'ranked_dyn_results' in locals() else []),
    ('Dynamic (Per Walk)', ranked_results_dynamic if 'ranked_results_dynamic' in locals() else []),
    ('MCMC', ranked_mh if 'ranked_mh' in locals() else [])
]

for name, lst in lists_to_process:
    if lst:
        for item in lst:
            total_visits[item['Node ID']] += item.get('Visit Count', 0)

# Get the Top 15 most visited nodes overall
top_nodes = [node for node, count in total_visits.most_common(15)]

# Build a subgraph containing shortest paths to these top nodes
vis_G = nx.DiGraph()
vis_G.add_node(target_id)

for n_id in top_nodes:
    try:
        # Find the shortest path from Bortezomib to the discovered node
        path = nx.shortest_path(graph_to_use, source=target_id, target=n_id)
        for i in range(len(path) - 1):
            u = path[i]
            v = path[i+1]
            vis_G.add_edge(u, v)
    except nx.NetworkXNoPath:
        continue

# Set up node colors, sizes, and labels based on their role
node_sizes = []
node_colors = []
labels = {}

for node in vis_G.nodes():
    name = graph_to_use.nodes[node].get('name', 'Unknown')
    labels[node] = name

    if node == target_id:
        # Target node (Bortezomib)
        node_sizes.append(2500)
        node_colors.append('#98FB98') # Light Green
    elif node in total_visits:
        # Discovered Disease/Side Effect (Size scaled by total visits)
        count = total_visits[node]
        # Base size 600, scale up by count, max out at 4000
        size = min(4000, 600 + (count * 0.5))
        node_sizes.append(size)
        node_colors.append('#87CEFA') # Sky Blue
    else:
        # Intermediate path node (Genes, etc.)
        node_sizes.append(400)
        node_colors.append('#D3D3D3') # Light Grey

# Draw the graph
plt.figure(figsize=(16, 12))
# spring_layout with k controls the distance between nodes
pos = nx.spring_layout(vis_G, seed=42, k=0.7)

nx.draw_networkx_nodes(vis_G, pos, node_size=node_sizes, node_color=node_colors, edgecolors='black', linewidths=1.5)
nx.draw_networkx_edges(vis_G, pos, alpha=0.6, arrows=True, arrowsize=15, width=1.5)
nx.draw_networkx_labels(vis_G, pos, labels, font_size=10, font_weight='bold')

plt.title("Aggregated Top Discoveries from Walk Algorithms", fontsize=18, fontweight='bold', pad=20)

# Create a custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Target (Bortezomib)', markerfacecolor='#98FB98', markersize=15, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', label='Top Discovery (Size = Visits)', markerfacecolor='#87CEFA', markersize=15, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', label='Intermediate Node', markerfacecolor='#D3D3D3', markersize=10, markeredgecolor='black')
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=12)

plt.axis('off')
plt.tight_layout()
plt.show()